# Existing iso-surfaces versus Ferreus RMT on real geochemistry data

This notebook compares the existing surface_apps.iso_surfaces marching-cubes extraction with ferreus_rmt using the geochem_pts dataset.

The comparison deliberately holds the interpolated scalar field constant:

1. the existing application interpolates Cu_ppm observations onto a regular grid;
2. marching cubes extracts a surface from that grid; and
3. Ferreus RMT extracts the same isovalue from a trilinear function over that same grid.

Consequently, differences measured here are primarily differences between the mesh extractors.

In [ ]:
from datetime import datetime
from pathlib import Path
from time import perf_counter

from ferreus_rmt import BoundaryClosure, ClusterMethod, build_isosurface
from geoh5py.groups import ContainerGroup
from geoh5py.objects import Points, Surface
from geoh5py.workspace import Workspace
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import RegularGridInterpolator
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy.spatial import cKDTree

from surface_apps.iso_surfaces.utils import (
    entity_to_grid,
    extract_iso_surfaces,
)

## Configuration

Cu_ppm = 200 was selected as a useful first anomaly boundary: it is above most observations but produces enough geometry for a meaningful comparison. These are experiment parameters, not universally correct geological thresholds.

The 50 m grid resolution and 500 m maximum interpolation distance match the existing application's UI defaults. A grid point farther than the maximum distance from data remains unsupported (NaN).

In [ ]:
input_candidates = [
    Path.cwd() / "geochem_pts" / "geochem_pts_example.geoh5",
    Path.cwd()
    / "surface_apps"
    / "ferreus"
    / "geochem_pts"
    / "geochem_pts_example.geoh5",
]

INPUT_PATH = next(
    (candidate for candidate in input_candidates if candidate.exists()),
    None,
)
if INPUT_PATH is None:
    raise FileNotFoundError(
        "Could not find geochem_pts/geochem_pts_example.geoh5 "
        "from the current working directory."
    )

OBJECT_NAME = "geochem_pts"
DATA_NAME = "Cu_ppm"
CONTOUR_LEVEL = 200.0
RESOLUTION = 50.0
MAX_DISTANCE = 500.0
RMT_RESOLUTION = RESOLUTION
RMT_CLUSTER_METHOD = ClusterMethod.CurvatureWeighted

OUTPUT_DIRECTORY = INPUT_PATH.parent / "ferreus_output"
OUTPUT_DIRECTORY.mkdir(exist_ok=True)

print("Input:", INPUT_PATH)
print("Field and isovalue:", DATA_NAME, CONTOUR_LEVEL)

## Load the observations and create the shared field

entity_to_grid is the existing application's preprocessing function. For a points object, it uses distance-weighted averaging to estimate the geochemical value at regular-grid locations.

This conversion is shared by both methods below.

In [ ]:
grid_start = perf_counter()

with Workspace(INPUT_PATH) as input_workspace:
    source_object = input_workspace.get_entity(OBJECT_NAME)[0]
    source_data = input_workspace.get_entity(DATA_NAME)[0]

    if source_object is None or source_data is None:
        raise ValueError(
            f"Could not find {OBJECT_NAME!r} and {DATA_NAME!r}."
        )

    source_locations = np.asarray(
        source_object.locations,
        dtype=np.float64,
    ).copy()
    source_values = np.asarray(
        source_data.values,
        dtype=np.float64,
    ).copy()

    comparison_grid, comparison_grid_values = entity_to_grid(
        source_object,
        source_data,
        resolution=RESOLUTION,
        max_distance=MAX_DISTANCE,
        horizon=None,
    )

grid_seconds = perf_counter() - grid_start

finite_source_values = source_values[np.isfinite(source_values)]
finite_grid = np.isfinite(comparison_grid_values)

print("Observation count:", source_locations.shape[0])
print("XYZ span:", np.ptp(source_locations, axis=0))
print("Finite Cu observations:", finite_source_values.size)
print("Cu observation percentiles:", np.percentile(
    finite_source_values,
    [10, 25, 50, 75, 90, 95, 99],
))
print("Grid shape:", comparison_grid_values.shape)
print(
    "Supported grid nodes:",
    int(np.count_nonzero(finite_grid)),
    "/",
    comparison_grid_values.size,
)
print("Shared gridding time (seconds):", grid_seconds)

## Existing method: marching cubes

This calls the existing application's extraction helper directly. Its result is the baseline users currently receive after the gridding step.

In [ ]:
marching_cubes_start = perf_counter()

existing_surfaces = extract_iso_surfaces(
    source_object,
    comparison_grid,
    [CONTOUR_LEVEL],
    comparison_grid_values,
)

marching_cubes_seconds = perf_counter() - marching_cubes_start

mc_vertices = np.asarray(existing_surfaces[0][0], dtype=np.float64)
mc_cells = np.asarray(existing_surfaces[0][1], dtype=np.uint32)

print("Marching-cubes vertices:", mc_vertices.shape)
print("Marching-cubes triangles:", mc_cells.shape)
print("Marching-cubes extraction time (seconds):", marching_cubes_seconds)

## Ferreus RMT on the same regular-grid field

RMT expects a callable scalar function instead of a dense value array. RegularGridInterpolator turns the existing grid into that function using trilinear interpolation. Unsupported regions remain NaN.

As a surface-following method, RMT also requires seed points near each surface component. The helper below independently finds regular-grid cells whose eight finite corner values straddle the requested isovalue, then uses their centres as seeds. It does not use the marching-cubes result. This seed search scans the dense grid.

In [ ]:
grid_interpolator = RegularGridInterpolator(
    tuple(comparison_grid),
    comparison_grid_values,
    method="linear",
    bounds_error=False,
    fill_value=np.nan,
)


def shared_grid_field(targets: np.ndarray) -> np.ndarray:
    """Evaluate the same trilinear scalar field represented by the grid."""
    return np.asarray(grid_interpolator(targets), dtype=np.float64)


def find_crossing_cell_seeds(
    grid: list[np.ndarray],
    grid_values: np.ndarray,
    isovalue: float,
) -> np.ndarray:
    """Return centres of fully supported grid cells crossing an isovalue."""
    corner_values = np.stack(
        [
            grid_values[x_slice, y_slice, z_slice]
            for x_slice in (slice(None, -1), slice(1, None))
            for y_slice in (slice(None, -1), slice(1, None))
            for z_slice in (slice(None, -1), slice(1, None))
        ],
        axis=-1,
    )

    fully_supported = np.all(np.isfinite(corner_values), axis=-1)
    crosses_isovalue = (
        fully_supported
        & (np.min(corner_values, axis=-1) <= isovalue)
        & (np.max(corner_values, axis=-1) >= isovalue)
    )

    crossing_indices = np.argwhere(crosses_isovalue)
    if crossing_indices.size == 0:
        raise ValueError(f"No fully supported grid cells cross {isovalue}.")

    return np.column_stack(
        [
            0.5
            * (
                grid[axis][crossing_indices[:, axis]]
                + grid[axis][crossing_indices[:, axis] + 1]
            )
            for axis in range(3)
        ]
    ).astype(np.float64)

In [ ]:
rmt_total_start = perf_counter()

rmt_seed_points = find_crossing_cell_seeds(
    comparison_grid,
    comparison_grid_values,
    CONTOUR_LEVEL,
)
seed_detection_seconds = perf_counter() - rmt_total_start

rmt_extents = np.array(
    [
        comparison_grid[0][0],
        comparison_grid[1][0],
        comparison_grid[2][0],
        comparison_grid[0][-1],
        comparison_grid[1][-1],
        comparison_grid[2][-1],
    ],
    dtype=np.float64,
)

rmt_extraction_start = perf_counter()
rmt_mesh = build_isosurface(
    seed_points=rmt_seed_points,
    extents=rmt_extents,
    resolution=RMT_RESOLUTION,
    isovalue=CONTOUR_LEVEL,
    surface_fn=shared_grid_field,
    cluster_method=RMT_CLUSTER_METHOD,
    boundary_closure=BoundaryClosure.None_,
)
rmt_extraction_seconds = perf_counter() - rmt_extraction_start
rmt_total_seconds = perf_counter() - rmt_total_start

rmt_vertices = np.asarray(rmt_mesh.vertices, dtype=np.float64)
rmt_cells = np.asarray(rmt_mesh.facets, dtype=np.uint32)

print("Independent RMT seed points:", rmt_seed_points.shape)
print("RMT vertices:", rmt_vertices.shape)
print("RMT triangles:", rmt_cells.shape)
print("Seed detection time (seconds):", seed_detection_seconds)
print("RMT extraction time (seconds):", rmt_extraction_seconds)
print("RMT seed plus extraction time (seconds):", rmt_total_seconds)

## Compare mesh quality, topology, and field agreement

Triangle quality is 1.0 for an equilateral triangle and approaches zero for increasingly skinny triangles. An edge shared by more than two triangles is non-manifold. Boundary edges can occur where a surface meets an unsupported region or the extraction bounds.

There is no known true copper-anomaly surface in this dataset, so this notebook cannot report geological accuracy. Instead it measures how closely each mesh follows the shared interpolated 200 ppm field and how closely the two meshes agree spatially.

In [ ]:
def calculate_mesh_statistics(
    mesh_vertices: np.ndarray,
    mesh_cells: np.ndarray,
    extraction_seconds: float,
) -> tuple[dict[str, float | int], np.ndarray]:
    """Calculate geometric, topological, and scalar-field diagnostics."""
    triangles = mesh_vertices[mesh_cells]
    edge_01 = triangles[:, 1] - triangles[:, 0]
    edge_12 = triangles[:, 2] - triangles[:, 1]
    edge_20 = triangles[:, 0] - triangles[:, 2]

    twice_area = np.linalg.norm(np.cross(edge_01, -edge_20), axis=1)
    squared_edge_sum = (
        np.sum(edge_01**2, axis=1)
        + np.sum(edge_12**2, axis=1)
        + np.sum(edge_20**2, axis=1)
    )
    triangle_quality = np.divide(
        2.0 * np.sqrt(3.0) * twice_area,
        squared_edge_sum,
        out=np.zeros_like(twice_area),
        where=squared_edge_sum > 0.0,
    )

    mesh_edges = np.vstack(
        [
            mesh_cells[:, [0, 1]],
            mesh_cells[:, [1, 2]],
            mesh_cells[:, [2, 0]],
        ]
    )
    mesh_edges.sort(axis=1)
    unique_edges, edge_use_counts = np.unique(
        mesh_edges,
        axis=0,
        return_counts=True,
    )

    graph = coo_matrix(
        (
            np.ones(unique_edges.shape[0] * 2),
            (
                np.r_[unique_edges[:, 0], unique_edges[:, 1]],
                np.r_[unique_edges[:, 1], unique_edges[:, 0]],
            ),
        ),
        shape=(mesh_vertices.shape[0], mesh_vertices.shape[0]),
    )
    component_count = connected_components(
        graph,
        directed=False,
        return_labels=False,
    )

    field_values = shared_grid_field(mesh_vertices)
    finite_field = np.isfinite(field_values)
    isovalue_errors = np.abs(field_values[finite_field] - CONTOUR_LEVEL)

    statistics = {
        "vertices": int(mesh_vertices.shape[0]),
        "triangles": int(mesh_cells.shape[0]),
        "extraction_seconds": float(extraction_seconds),
        "surface_area": float(np.sum(0.5 * twice_area)),
        "quality_mean": float(np.mean(triangle_quality)),
        "quality_5th_percentile": float(
            np.percentile(triangle_quality, 5)
        ),
        "components": int(component_count),
        "boundary_edges": int(np.count_nonzero(edge_use_counts == 1)),
        "non_manifold_edges": int(np.count_nonzero(edge_use_counts > 2)),
        "finite_field_fraction": float(np.mean(finite_field)),
        "mean_absolute_isovalue_error": float(np.mean(isovalue_errors)),
        "maximum_absolute_isovalue_error": float(np.max(isovalue_errors)),
    }

    return statistics, field_values


mc_statistics, mc_field_values = calculate_mesh_statistics(
    mc_vertices,
    mc_cells,
    marching_cubes_seconds,
)
rmt_statistics, rmt_field_values = calculate_mesh_statistics(
    rmt_vertices,
    rmt_cells,
    rmt_extraction_seconds,
)

comparison_rows = [
    ("Vertices", "vertices", ".0f"),
    ("Triangles", "triangles", ".0f"),
    ("Extraction seconds", "extraction_seconds", ".4f"),
    ("Surface area", "surface_area", ".1f"),
    ("Mean triangle quality", "quality_mean", ".4f"),
    ("5th-percentile triangle quality", "quality_5th_percentile", ".4f"),
    ("Connected components", "components", ".0f"),
    ("Boundary edges", "boundary_edges", ".0f"),
    ("Non-manifold edges", "non_manifold_edges", ".0f"),
    ("Finite field-value fraction", "finite_field_fraction", ".4f"),
    ("Mean absolute isovalue error", "mean_absolute_isovalue_error", ".4f"),
    ("Maximum absolute isovalue error", "maximum_absolute_isovalue_error", ".4f"),
]

print(f"{'Metric':40s} {'Marching cubes':>18s} {'Ferreus RMT':>18s}")
print("-" * 78)
for label, key, value_format in comparison_rows:
    mc_text = format(mc_statistics[key], value_format)
    rmt_text = format(rmt_statistics[key], value_format)
    print(f"{label:40s} {mc_text:>18s} {rmt_text:>18s}")




### Direct positional agreement

Nearest-vertex distances provide an approximate comparison between the meshes. They are not exact point-to-triangle distances, so interpret them relative to the configured grid resolution.

In [ ]:
mc_to_rmt_distances = cKDTree(rmt_vertices).query(mc_vertices)[0]
rmt_to_mc_distances = cKDTree(mc_vertices).query(rmt_vertices)[0]
symmetric_distances = np.r_[mc_to_rmt_distances, rmt_to_mc_distances]

print("Marching cubes to RMT mean distance:", mc_to_rmt_distances.mean())
print("RMT to marching cubes mean distance:", rmt_to_mc_distances.mean())
print(
    "Symmetric distance percentiles (50%, 90%, 95%, 100%):",
    np.percentile(symmetric_distances, [50, 90, 95, 100]),
)

### Reading the result

Compare triangle quality, component counts, field error, and positional separation together. A method can produce attractive triangles while missing small components, or reproduce the isovalue accurately while retaining questionable interpolation artefacts. Because this dataset has no known true 200 ppm surface, neither output can be labelled geologically correct from these metrics alone.

## Visual comparison

The source points shown are finite observations at or above 200 ppm. They are not expected to lie exactly on the interpolated surface: the surface encloses/intersects regions where the continuous estimated field equals 200 ppm.

Matplotlib's 3D renderer seems to have issues even when I tried to limit the complexity of what it's displaying, so I couldn't get this to work and it just keeps working for minutes.

In [ ]:
'''MAX_PREVIEW_TRIANGLES = 2_500


def select_preview_cells(
    cells: np.ndarray,
    maximum_triangles: int = MAX_PREVIEW_TRIANGLES,
) -> np.ndarray:
    """Select evenly spaced triangles for responsive notebook rendering."""
    if cells.shape[0] <= maximum_triangles:
        return cells

    preview_indices = np.linspace(
        0,
        cells.shape[0] - 1,
        maximum_triangles,
        dtype=int,
    )
    return cells[preview_indices]


high_copper = np.isfinite(source_values) & (source_values >= CONTOUR_LEVEL)
coordinate_span = np.ptp(source_locations, axis=0)

figure = plt.figure(figsize=(15, 7))
plot_meshes = [
    ("Existing marching cubes", mc_vertices, mc_cells, "darkorange"),
    ("Ferreus RMT", rmt_vertices, rmt_cells, "steelblue"),
]

for plot_index, (title, vertices, full_cells, colour) in enumerate(
    plot_meshes,
    start=1,
):
    preview_cells = select_preview_cells(full_cells)
    print(
        f"Rendering {title}: {preview_cells.shape[0]:,} of "
        f"{full_cells.shape[0]:,} triangles"
    )

    axis = figure.add_subplot(1, 2, plot_index, projection="3d")
    axis.plot_trisurf(
        vertices[:, 0],
        vertices[:, 1],
        vertices[:, 2],
        triangles=preview_cells,
        color=colour,
        linewidth=0.0,
        antialiased=False,
        shade=False,
        alpha=1.0,
    )
    axis.scatter(
        source_locations[high_copper, 0],
        source_locations[high_copper, 1],
        source_locations[high_copper, 2],
        color="black",
        s=5,
        alpha=0.6,
        label=f"Observed {DATA_NAME} >= {CONTOUR_LEVEL:g}",
    )
    axis.set_box_aspect(coordinate_span)
    axis.set_xlabel("X")
    axis.set_ylabel("Y")
    axis.set_zlabel("Z")
    axis.view_init(elev=22, azim=-55)
    axis.set_title(title)
    axis.legend(loc="upper right")

figure.suptitle(f"{DATA_NAME} = {CONTOUR_LEVEL:g} ppm")
figure.subplots_adjust(left=0.02, right=0.98, bottom=0.02, top=0.90)
plt.show()'''

## Export both methods to geoh5

The output includes a copy of the geochemistry points and both surfaces. Each mesh has its interpolated field value and nearest-vertex distance to the other method, making it possible to inspect differences in Geoscience ANALYST.

In [ ]:
# Recreate the output directory if previous results were deleted after
# the configuration cell was run.
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

output_path = OUTPUT_DIRECTORY / (
    f"real_geochem_mc_vs_rmt_{datetime.now():%Y%m%d_%H%M%S}.geoh5"
)

with Workspace.create(output_path) as output_workspace:
    result_group = ContainerGroup.create(
        output_workspace,
        name=f"{DATA_NAME} {CONTOUR_LEVEL:g} - MC versus RMT",
    )

    output_points = Points.create(
        output_workspace,
        name=OBJECT_NAME,
        vertices=source_locations,
        parent=result_group,
    )
    output_points.add_data({DATA_NAME: {"values": source_values}})

    output_mc_surface = Surface.create(
        output_workspace,
        name=f"Existing marching cubes - {CONTOUR_LEVEL:g} ppm",
        vertices=mc_vertices,
        cells=mc_cells,
        parent=result_group,
    )
    output_mc_surface.add_data(
        {
            "Interpolated Cu ppm": {"values": mc_field_values},
            "Nearest RMT vertex distance": {
                "values": mc_to_rmt_distances
            },
        }
    )

    output_rmt_surface = Surface.create(
        output_workspace,
        name=f"Ferreus RMT - {CONTOUR_LEVEL:g} ppm",
        vertices=rmt_vertices,
        cells=rmt_cells,
        parent=result_group,
    )
    output_rmt_surface.add_data(
        {
            "Interpolated Cu ppm": {"values": rmt_field_values},
            "Nearest marching-cubes vertex distance": {
                "values": rmt_to_mc_distances
            },
        }
    )

print("Created:", output_path)

## Interpretation limits and next tests

A successful run establishes that RMT can extract real, disconnected geochemical anomaly surfaces from the same field used by the current application, including in the presence of unsupported regions.

It does not show that either surface is geological truth. Both inherit assumptions and artefacts from the shared weighted-average grid. It also does not test the RBF package or constraint fitting.

Useful follow-up experiments are:

- repeat at several contour levels and resolutions;
- compare results at approximately equal triangle count or positional error;
- test RMT boundary-closure modes on a genuinely enclosed component;
- replace grid-cell seed detection with seeds derived from selected observations or another production-safe strategy; and
- separately compare the existing weighted-average field with a ferreus_rbf field fitted from the raw geochemistry points.